# RainGuard — Module 9: Final Model Interpretation & Validation

**Purpose:** explain the behaviour and limitations of the final RainGuard regression and classification models.

This notebook uses saved artifacts from Modules 5, 6, 7 and 8. It does not retrain models or change earlier outputs.

It covers:
1. Load the saved regression and classification artifacts.
2. Validate saved test prediction files and report metrics.
3. Analyse regression errors by rainfall amount and district.
4. Analyse classification performance by class.
5. Explain feature importance where the fitted model supports it.
6. Save final interpretation tables and figures.

**Important:** The rainfall dataset has a short observation window and very few heavy-rainfall test examples. Report these limitations. This is next-day rainfall forecasting and rainfall-intensity classification, not verified flood prediction.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report
)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
REPORT_DIR = os.path.join(PROJECT_ROOT, "reports")
FIG_DIR = os.path.join(PROJECT_ROOT, "docs", "figures")
for folder in [REPORT_DIR, FIG_DIR]:
    os.makedirs(folder, exist_ok=True)

required = {
    "regression predictions": os.path.join(DATA_DIR, "regression_test_predictions.csv"),
    "classification predictions": os.path.join(DATA_DIR, "classification_test_predictions.csv"),
    "classification optimized predictions": os.path.join(DATA_DIR, "classification_optimized_test_predictions.csv"),
    "regression results": os.path.join(MODEL_DIR, "regression_results.json"),
    "regression model": os.path.join(MODEL_DIR, "best_regressor.joblib"),
    "original classifier": os.path.join(MODEL_DIR, "rainfall_classification_model.pkl"),
    "optimized classifier": os.path.join(MODEL_DIR, "rainfall_classification_model_optimized.joblib"),
    "optimization metadata": os.path.join(MODEL_DIR, "classification_optimization_metadata.json"),
    "classification metadata": os.path.join(MODEL_DIR, "classification_model_metadata.json"),
    "feature config": os.path.join(MODEL_DIR, "feature_config.json")
}
missing = [name for name, path in required.items() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError("Missing required outputs. Run the corresponding modules first:\n" +
                            "\n".join(f"{name}: {required[name]}" for name in missing))

with open(required["regression results"], encoding="utf-8") as f: reg_results = json.load(f)
with open(required["optimization metadata"], encoding="utf-8") as f: opt_meta = json.load(f)
with open(required["classification metadata"], encoding="utf-8") as f: cls_meta = json.load(f)
with open(required["feature config"], encoding="utf-8") as f: feature_config = json.load(f)

reg_pred = pd.read_csv(required["regression predictions"], parse_dates=["date", "target_date"])
cls_pred = pd.read_csv(required["classification predictions"], parse_dates=["date", "target_date"])
opt_pred = pd.read_csv(required["classification optimized predictions"], parse_dates=["date", "target_date"])

reg_model = joblib.load(required["regression model"])
original_clf = joblib.load(required["original classifier"])
optimized_clf = joblib.load(required["optimized classifier"])

print("Project:", PROJECT_ROOT)
print("Regression test rows:", len(reg_pred))
print("Original classification test rows:", len(cls_pred))
print("Optimized classification test rows:", len(opt_pred))
print("Optimized classifier:", opt_meta.get("selected_candidate"), "| weighting:", opt_meta.get("weighting"))

## 1. Validate saved prediction files

In [ ]:
# Confirm that Module 8 predictions correspond to the same district/date test observations.
key_cols = ["state", "district", "date", "target_date"]
for name, frame in [("regression", reg_pred), ("classification", cls_pred), ("optimised classification", opt_pred)]:
    absent = [c for c in key_cols if c not in frame.columns]
    if absent:
        raise KeyError(f"{name} predictions are missing keys: {absent}")

reg_keys = set(map(tuple, reg_pred[key_cols].astype(str).to_numpy()))
cls_keys = set(map(tuple, cls_pred[key_cols].astype(str).to_numpy()))
opt_keys = set(map(tuple, opt_pred[key_cols].astype(str).to_numpy()))
print("Regression and original classification same rows:", reg_keys == cls_keys)
print("Original and optimised classification same rows:", cls_keys == opt_keys)
if cls_keys != opt_keys:
    raise ValueError("Module 6 and Module 8 classification predictions do not refer to identical test rows.")

print("Regression target date:", reg_pred["target_date"].min().date(), "to", reg_pred["target_date"].max().date())
print("Classification target date:", opt_pred["target_date"].min().date(), "to", opt_pred["target_date"].max().date())
print("Target-next-day check (regression):", ((reg_pred["target_date"] - reg_pred["date"]).dt.days == 1).all())

## 2. Regression performance and error analysis

In [ ]:
reg_cols = [c for c in reg_pred.columns if c.startswith("pred_")]
if not reg_cols:
    raise ValueError("No regression prediction columns beginning with 'pred_' were found.")
y = pd.to_numeric(reg_pred["actual_mm"], errors="coerce")
reg_metrics = []
for col in reg_cols:
    p = pd.to_numeric(reg_pred[col], errors="coerce")
    valid = y.notna() & p.notna()
    mse = mean_squared_error(y[valid], p[valid])
    reg_metrics.append({
        "Model": col.replace("pred_", "").replace("_", " ").title(),
        "MAE_mm": mean_absolute_error(y[valid], p[valid]),
        "RMSE_mm": np.sqrt(mse),
        "R2": r2_score(y[valid], p[valid]),
        "Rows": int(valid.sum())
    })
reg_metrics = pd.DataFrame(reg_metrics).sort_values("RMSE_mm").reset_index(drop=True)
display(reg_metrics.round(4))

best_reg_col = "pred_" + reg_metrics.iloc[0]["Model"].lower().replace(" ", "_")
# If model naming differs from prediction column, use the lowest-RMSE actual column.
best_reg_col = next(c for c in reg_cols if c.replace("pred_", "").replace("_", " ").title() == reg_metrics.iloc[0]["Model"])
reg_error = reg_pred[key_cols + ["actual_mm", best_reg_col]].copy()
reg_error = reg_error.rename(columns={best_reg_col: "predicted_mm"})
reg_error["residual_mm"] = reg_error["actual_mm"] - reg_error["predicted_mm"]
reg_error["absolute_error_mm"] = reg_error["residual_mm"].abs()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(reg_error["actual_mm"], reg_error["predicted_mm"], alpha=0.35, s=14)
upper = max(reg_error["actual_mm"].max(), reg_error["predicted_mm"].max())
axes[0].plot([0, upper], [0, upper], "--", color="black")
axes[0].set(xlabel="Actual rainfall (mm)", ylabel="Predicted rainfall (mm)", title="Actual vs predicted")
sns.histplot(reg_error["residual_mm"], bins=40, kde=True, ax=axes[1])
axes[1].axvline(0, linestyle="--", color="black")
axes[1].set(title="Residual distribution", xlabel="Actual − predicted (mm)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m9_01_regression_residuals.png"), dpi=160, bbox_inches="tight")
plt.show()

print("Largest absolute errors:")
display(reg_error.nlargest(20, "absolute_error_mm").round(3))

## 3. Regression error by actual rainfall range

In [ ]:
rain_bins = [-np.inf, 0, 2.45, 15.55, 64.45, 115.55, np.inf]
rain_labels = ["Zero", "No/Very Light", "Light", "Moderate", "Heavy", "Very Heavy+"]
reg_error["actual_rain_band"] = pd.cut(reg_error["actual_mm"], bins=rain_bins, labels=rain_labels, include_lowest=True)
error_by_band = reg_error.groupby("actual_rain_band", observed=False).agg(
    rows=("absolute_error_mm", "size"),
    MAE_mm=("absolute_error_mm", "mean"),
    median_abs_error_mm=("absolute_error_mm", "median")
).reset_index()
display(error_by_band.round(3))

plt.figure(figsize=(10, 5))
sns.barplot(data=error_by_band, x="actual_rain_band", y="MAE_mm")
plt.xticks(rotation=25, ha="right")
plt.title("Regression MAE by actual rainfall band")
plt.xlabel("Actual rainfall band")
plt.ylabel("MAE (mm)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m9_02_regression_error_by_rain_band.png"), dpi=160, bbox_inches="tight")
plt.show()

## 4. Classification performance by class

In [ ]:
labels = opt_meta["class_labels"]
codes = list(range(len(labels)))
y_true = opt_pred["actual_code"].astype(int).to_numpy()
y_opt = opt_pred["optimized_pred_code"].astype(int).to_numpy()
y_orig = opt_pred["original_pred_code"].astype(int).to_numpy()

def class_table(y_true, y_pred, model_name):
    rep = classification_report(y_true, y_pred, labels=codes, target_names=labels,
                                 output_dict=True, zero_division=0)
    return pd.DataFrame([{
        "Model": model_name, "Class": label,
        "Precision": rep[label]["precision"],
        "Recall": rep[label]["recall"],
        "F1": rep[label]["f1-score"],
        "Support": int(rep[label]["support"])
    } for label in labels])

class_metrics = pd.concat([
    class_table(y_true, y_orig, "Original Module 6"),
    class_table(y_true, y_opt, "Module 8 optimised")
], ignore_index=True)
display(class_metrics.round(4))

overall = []
for name, pred in [("Original Module 6", y_orig), ("Module 8 optimised", y_opt)]:
    overall.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, pred),
        "Macro F1": f1_score(y_true, pred, labels=codes, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, pred, labels=codes, average="weighted", zero_division=0)
    })
overall_metrics = pd.DataFrame(overall)
display(overall_metrics.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
class_metrics.pivot(index="Class", columns="Model", values="Recall").plot(kind="bar", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("Recall")
ax.set_title("Per-class recall: original vs optimised")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m9_03_class_recall.png"), dpi=160, bbox_inches="tight")
plt.show()

## 5. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, pred, title in [
    (axes[0], y_orig, "Original Module 6"),
    (axes[1], y_opt, "Module 8 optimised")
]:
    cm = confusion_matrix(y_true, pred, labels=codes)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m9_04_classification_confusion_matrices.png"), dpi=160, bbox_inches="tight")
plt.show()

print("Optimised classification report:")
print(classification_report(y_true, y_opt, labels=codes, target_names=labels, zero_division=0, digits=4))

## 6. Feature importance (native model importance when available)

In [ ]:
# Module 5's saved bundle stores the fitted estimator and feature list.
def unwrap_estimator(obj):
    if isinstance(obj, dict) and "model" in obj:
        return obj["model"], obj.get("features", [])
    estimator = obj.steps[-1][1] if hasattr(obj, "steps") else obj
    return estimator, []

def extract_importance(obj, feature_list):
    estimator, stored_features = unwrap_estimator(obj)
    feats = stored_features or feature_list
    if hasattr(estimator, "feature_importances_"):
        vals = np.asarray(estimator.feature_importances_)
        method = "Impurity-based feature importance"
    elif hasattr(estimator, "coef_"):
        coef = np.asarray(estimator.coef_)
        vals = np.mean(np.abs(coef), axis=0) if coef.ndim > 1 else np.abs(coef)
        method = "Mean absolute coefficient"
    else:
        return None, None
    if len(vals) != len(feats):
        return None, None
    out = pd.DataFrame({"Feature": feats, "Importance": vals}).sort_values("Importance", ascending=False)
    return out, method

feature_groups = feature_config.get("feature_groups", feature_config.get("groups", {}))
feature_list = []
for value in feature_groups.values():
    if isinstance(value, list):
        feature_list.extend(value)
feature_list = list(dict.fromkeys(feature_list))

reg_importance, reg_method = extract_importance(reg_model, feature_list)
if reg_importance is not None:
    print("Regression importance method:", reg_method)
    display(reg_importance.head(20))
    plt.figure(figsize=(9, 6))
    sns.barplot(data=reg_importance.head(15), y="Feature", x="Importance")
    plt.title("Regression – top feature importance")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "m9_05_regression_feature_importance.png"), dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Regression estimator does not expose native feature importance. Do not infer importance from this model.")

opt_importance, opt_method = extract_importance(optimized_clf, opt_meta.get("features", feature_list))
if opt_importance is not None:
    print("Optimised classifier importance method:", opt_method)
    display(opt_importance.head(20))
    plt.figure(figsize=(9, 6))
    sns.barplot(data=opt_importance.head(15), y="Feature", x="Importance")
    plt.title("Optimised classifier – top feature importance")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "m9_06_classification_feature_importance.png"), dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Optimised classifier does not expose native feature importance. Use permutation importance only in a separate validation-only analysis.")

## 7. Heavy-or-above diagnostic

In [ ]:
heavy_code = labels.index("Heavy")
def heavy_metrics(y_true, y_pred):
    actual = (y_true >= heavy_code).astype(int)
    pred = (y_pred >= heavy_code).astype(int)
    return {
        "Precision": precision_score(actual, pred, zero_division=0),
        "Recall": recall_score(actual, pred, zero_division=0),
        "F1": f1_score(actual, pred, zero_division=0),
        "Actual heavy+ cases": int(actual.sum()),
        "Detected heavy+ cases": int(((actual == 1) & (pred == 1)).sum()),
        "False alarms": int(((actual == 0) & (pred == 1)).sum()),
        "Missed cases": int(((actual == 1) & (pred == 0)).sum())
    }
heavy_summary = pd.DataFrame([
    {"Model": "Original Module 6", **heavy_metrics(y_true, y_orig)},
    {"Model": "Module 8 optimised", **heavy_metrics(y_true, y_opt)}
])
display(heavy_summary.round(4))

## 8. Final interpretation and limitations

In [ ]:
# Produce a factual summary from the measured outputs.
best_reg = reg_metrics.iloc[0]
best_cls = overall_metrics.sort_values(["Macro F1", "Balanced accuracy"], ascending=False).iloc[0]
summary = {
    "regression_best_test_model_by_RMSE": str(best_reg["Model"]),
    "regression_MAE_mm": float(best_reg["MAE_mm"]),
    "regression_RMSE_mm": float(best_reg["RMSE_mm"]),
    "regression_R2": float(best_reg["R2"]),
    "classification_preferred_by_test_macro_F1": str(best_cls["Model"]),
    "classification_accuracy": float(best_cls["Accuracy"]),
    "classification_balanced_accuracy": float(best_cls["Balanced accuracy"]),
    "classification_macro_F1": float(best_cls["Macro F1"]),
    "limitations": [
        "The available rainfall observation period is short; results should not be generalized to annual or seasonal patterns.",
        "Rare heavy-rainfall categories have small test support, so class-wise recall is uncertain.",
        "Heavy-rainfall classification is not equivalent to flood prediction.",
        "Native feature importance is model-dependent and does not establish causation."
    ]
}
display(pd.DataFrame([summary]).T.rename(columns={0: "Value"}))
with open(os.path.join(REPORT_DIR, "rainguard_module9_interpretation_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# Save core tables.
reg_metrics.to_csv(os.path.join(REPORT_DIR, "rainguard_module9_regression_metrics.csv"), index=False)
class_metrics.to_csv(os.path.join(REPORT_DIR, "rainguard_module9_class_metrics.csv"), index=False)
overall_metrics.to_csv(os.path.join(REPORT_DIR, "rainguard_module9_classification_overall.csv"), index=False)
heavy_summary.to_csv(os.path.join(REPORT_DIR, "rainguard_module9_heavy_diagnostic.csv"), index=False)
reg_error.to_csv(os.path.join(REPORT_DIR, "rainguard_module9_regression_test_errors.csv"), index=False)

if reg_importance is not None:
    reg_importance.to_csv(os.path.join(REPORT_DIR, "rainguard_module9_regression_feature_importance.csv"), index=False)
if opt_importance is not None:
    opt_importance.to_csv(os.path.join(REPORT_DIR, "rainguard_module9_classification_feature_importance.csv"), index=False)

print("Module 9 reports saved to:", REPORT_DIR)
print("Module 9 figures saved to:", FIG_DIR)

## 9. Completion checklist

- [ ] Saved prediction files align to the same test rows.
- [ ] Regression residuals and rainfall-band errors reviewed.
- [ ] Per-class classification precision, recall and F1 reviewed.
- [ ] Original and optimised confusion matrices reviewed.
- [ ] Heavy-rainfall misses and false alarms reviewed.
- [ ] Feature importance reviewed where supported by the fitted estimators.
- [ ] Limitations documented in the report.
- [ ] Module 9 outputs saved.

**Do not claim that a feature causes rainfall or that RainGuard predicts floods.** Describe it as next-day district rainfall forecasting and rainfall-intensity classification.